In [1]:
%load_ext autoreload
%autoreload 2
from innolux import *

In [2]:
# set polars show all rows
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [3]:
cwd = data_path / "T2 6117" / "pds"

In [8]:
cwd

WindowsPath('D:/Documents/Projects/innolux/data/T2 6117')

In [4]:
def edc_get(
    edc_raw: pl.DataFrame, 
    step: tuple[str, str], 
    filter: list[pl.Expr] | None = None,
) -> pl.DataFrame:
    result = (
        edc_raw.filter(
            pl.col("SUB_EQUIP_ID").str.contains(step[0]),
            pl.col("PARAM_NAME").str.contains(step[1])
        )
        # .select(
        #     "GLASS_ID", "SUB_EQUIP_ID", "PARAM_NAME", "AVG_VALUE"
        # )
        .select(
            "ID", 
            pl.col("PARAM_VALUE").cast(float)
        )
    )
    
    if filter is not None:
        result = result.filter(*filter)
    
    return result.group_by(['ID']).mean()

In [5]:
# lf - lazy data frame
eda_lf = pl.scan_csv(cwd / "pds-*.csv", ignore_errors=True)
eda_raw = (
    eda_lf
        .with_columns(
            pl.col("PARAM_VALUE")
                .cast(float, strict=False)
                .fill_null(0),
            pl.col("GLASS_ID").alias("ID"),
        )
        .filter(pl.col("PARAM_VALUE") > 0)
        .collect()
)

In [16]:
condition = pl.read_excel(cwd / "condition.xlsx")

In [17]:
condition

Condition,TFT,CF
str,str,str
"""LC-1098""","""611A2APB2HP221""","""611A2FPB2CBT15"""
"""LC-1098""","""611A2APB2HP208""","""611A2FPB2CBT23"""
"""STD-首件""","""611A2APB2HP213""","""611A2FPB2C9T27"""
"""STD""","""611A2APB2HP212""","""611A2FPB2CDT47"""
"""STD""","""611A2APB2HP216""","""611A2FPB2CET58"""
"""PAUV 260""","""611A2APB2HP204""","""611A2FPB2C8T35"""
"""PAUV 260""","""611A2APB2HP214""","""611A2FPB2CDT41"""
"""PAUV 220""","""611A2APB2HP206""","""611A2FPB2CGT19"""
"""PAUV 220""","""611A2APB2HP211""","""611A2FPB2CBT32"""


In [111]:
eda_lf.filter(pl.col("SUB_EQUIP_ID").str.contains(r"2CPHA340")).head().collect()

STEP_ID,GLASS_ID,PROCESS_START_TIME,PROCESS_END_TIME,COMPONENT_ID,COMPONENT_TYPE,EQUIP_ID,SUB_EQUIP_ID,PRODUCT_ID,BATCH_ID,LOT_ID,ARRAY_LOT_ID,ARRAY_GLASS_ID,CF_GLASS_ID,RECIPE_ID,TA_ID,CASSETTE_ID,PROCESS_MODE,PLAN_ID,FAB_AREA,BATCH_TYPE,ABNORMAL_FLAG,PANEL_SIZE,MODEL_NAME,SLOT_ID,PARAM_COLLECTION,PARAM_NAME,PARAM_VALUE,AVG,STD,MAX,MIN,COUNT,PARAM_STR_VALUE,RN,
str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str
"""6220_PH3_PA_UV""","""611A2APB2HP201""","""2026/01/22 17:23:43""","""2026/01/22 17:50:26""","""611A2APB2HP201""",1,"""2CPHA300""","""2CPHA340""","""611A2C""","""EF261216""","""611A2APB2HR2.02""","""611A2APB2HP2""","""611A2APB2HP201""",""" ""","""611A_T01""","""BC""","""2W278""","""IPS""","""IPSPHA1""","""T2""","""E""",""" ""","""6.1W""","""F061A17-6T1""",62,"""PDSG_2CPHA300_2CPHA340""","""1 HEAP differential""","""217""",""" """,""" """,""" """,""" """,""" """,""" """,1,null
"""6220_PH3_PA_UV""","""611A2APB2HP201""","""2026/01/22 17:23:43""","""2026/01/22 17:50:26""","""611A2APB2HP201""",1,"""2CPHA300""","""2CPHA340""","""611A2C""","""EF261216""","""611A2APB2HR2.02""","""611A2APB2HP2""","""611A2APB2HP201""",""" ""","""611A_T01""","""BC""","""2W278""","""IPS""","""IPSPHA1""","""T2""","""E""",""" ""","""6.1W""","""F061A17-6T1""",62,"""PDSG_2CPHA300_2CPHA340""","""1 OZONE""","""0""",""" """,""" """,""" """,""" """,""" """,""" """,2,null
"""6220_PH3_PA_UV""","""611A2APB2HP201""","""2026/01/22 17:23:43""","""2026/01/22 17:50:26""","""611A2APB2HP201""",1,"""2CPHA300""","""2CPHA340""","""611A2C""","""EF261216""","""611A2APB2HR2.02""","""611A2APB2HP2""","""611A2APB2HP201""",""" ""","""611A_T01""","""BC""","""2W278""","""IPS""","""IPSPHA1""","""T2""","""E""",""" ""","""6.1W""","""F061A17-6T1""",62,"""PDSG_2CPHA300_2CPHA340""","""1 differential pre.""","""1759""",""" """,""" """,""" """,""" """,""" """,""" """,3,null
"""6220_PH3_PA_UV""","""611A2APB2HP201""","""2026/01/22 17:23:43""","""2026/01/22 17:50:26""","""611A2APB2HP201""",1,"""2CPHA300""","""2CPHA340""","""611A2C""","""EF261216""","""611A2APB2HR2.02""","""611A2APB2HP2""","""611A2APB2HP201""",""" ""","""611A_T01""","""BC""","""2W278""","""IPS""","""IPSPHA1""","""T2""","""E""",""" ""","""6.1W""","""F061A17-6T1""",62,"""PDSG_2CPHA300_2CPHA340""","""1 input Wator temp.""","""21.2""",""" """,""" """,""" """,""" """,""" """,""" """,4,null
"""6220_PH3_PA_UV""","""611A2APB2HP201""","""2026/01/22 17:23:43""","""2026/01/22 17:50:26""","""611A2APB2HP201""",1,"""2CPHA300""","""2CPHA340""","""611A2C""","""EF261216""","""611A2APB2HR2.02""","""611A2APB2HP2""","""611A2APB2HP201""",""" ""","""611A_T01""","""BC""","""2W278""","""IPS""","""IPSPHA1""","""T2""","""E""",""" ""","""6.1W""","""F061A17-6T1""",62,"""PDSG_2CPHA300_2CPHA340""","""1 output wator temp.""","""33.5""",""" """,""" """,""" """,""" """,""" """,""" """,5,null


In [70]:
(
    eda_raw
        .filter(
            pl.col("ID") == "611A2APB2HP225",
            pl.col("SUB_EQUIP_ID").str.contains(r"2CASM\d+"),
            pl.col("PARAM_NAME").str.contains(r"ODFIN_UV_QTime"),
            # pl.col("PARAM_NAME").str.contains(r"Oven_\d_Temp\d"),
            # pl.col("PARAM_VALUE").is_between(500,700),
        )
        .select(
            "ID", 
            "SUB_EQUIP_ID",
            "PARAM_NAME",
            "PARAM_VALUE"
        )
        # .slice(110,20)
)

ID,SUB_EQUIP_ID,PARAM_NAME,PARAM_VALUE
str,str,str,f64
"""611A2APB2HP225""","""2CASM1C0""","""ODFIN_UV_QTime""",307.0


In [42]:
t2_edc_steps = {
    "PIPR TEMP": {"step": (r"2CPIL\d+", r"PreCur\d+_temp"), "filter": None},
    "PIPR TIME": {"step": (r"2CPIL\d+", r"PreCur\d+_Time"), "filter": None},
    "PIPB TEMP": {"step": (r"2CPIL\d+", r"Oven_Temp_\d+"), "filter": None},
    "PIPB TIME": {"step": (r"2CPIL\d+", r"Heating_Time"), "filter": None},
    "PAUV": {"step": (r"2CPHA\d+", r"\d-\d exposure"), "filter": None},
    "PAPB TEMP": {"step": (r"2CPHA\d+", r"Baking temp zone\d+"), "filter": None},
    "PAPB TIME": {"step": (r"2CPHA\d+", r"Heating_Time"), "filter": None},
    "BOOX TEMP": {"step": (r"2CASM\d+", r"Oven_Temp_\d+"), "filter": None},
    "BOOX TIME": {"step": (r"2CASM\d30", r"Heating_Time"), "filter": None},
    "SEUV": {"step": (r"2CASM\dC0", r"UV_Energy"), "filter": None},
    "SEPB TEMP": {"step": (r"2CASM\dD0", r"Temp"), "filter": None},
    "SEPB TIME": {"step": (r"2CASM\dD0", r"Heating_Time"), "filter": None},
}
eda = eda_raw.select("ID").unique()
for step, kwargs in t2_edc_steps.items():
    eda = eda.join(
        edc_get(eda_raw, **kwargs)
        .select(
            "ID",
            pl.col("PARAM_VALUE").alias(step),
        ),
        on='ID',
        how='left',
    )
eda

ID,PIPR TEMP,PIPR TIME,PIPB TEMP,PIPB TIME,PAUV,PAPB TEMP,PAPB TIME,BOOX TEMP,BOOX TIME,SEUV,SEPB TEMP,SEPB TIME
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""611A2APB2HP215""",135.0,80.0,210.966667,656.0,262.666667,230.33,1802.0,151.033333,608.0,600.0,120.0,3649.0
"""611A2APB2HP228""",135.0,80.0,210.733333,607.0,262.666667,230.05,1801.0,150.833333,604.0,700.0,119.98,3648.0
"""611A2FPB2CCT55""",135.0,80.0,210.516667,608.0,262.666667,230.24,1802.0,150.833333,604.0,null,null,null
"""611A2FPB2C8T46""",135.0,80.0,210.666667,607.0,263.333333,230.05,1802.0,151.033333,604.0,null,null,null
"""611A2APB2HP206""",135.0,80.0,210.933333,639.0,246.333333,230.01,1802.0,150.966667,604.0,600.0,119.98,3649.0
"""611A2FPB2CCT50""",135.0,80.0,210.483333,608.0,262.666667,230.37,1804.0,151.166667,604.0,null,null,null
"""611A2FPB2CGT16""",135.0,80.0,211.0,608.0,262.666667,230.42,1803.0,150.966667,604.0,null,null,null
"""611A2APB2HP219""",135.0,80.0,210.633333,607.0,263.0,230.42,1801.0,150.966667,604.0,600.0,119.98,3648.0
"""611A2FPB2CGT19""",135.0,80.0,210.766667,607.0,246.333333,230.13,1802.0,150.966667,604.0,null,null,null


In [43]:
(condition
    .join(
        eda.select([col for col in eda.columns if eda[col].null_count() == 0]),
        left_on="CF",
        right_on='ID',
        suffix="_CF",
        maintain_order="left",
    )
    .join(
        eda,
        left_on="TFT",
        right_on="ID",
        suffix=" - T",
        maintain_order="left",
    )
    .select(
        pl.col('Condition'),
        pl.col('TFT'),
        pl.col('CF'),
        pl.col('PIPR TEMP - T'),
        pl.col('PIPR TEMP').alias('PIPR TEMP - C'),
        pl.col('PIPR TIME - T'),
        pl.col('PIPR TIME').alias('PIPR TIME - C'),
        pl.col('PIPB TEMP - T'),
        pl.col('PIPB TEMP').alias('PIPB TEMP - C'),
        pl.col('PIPB TIME - T'),
        pl.col('PIPB TIME').alias('PIPB TIME - C'),
        pl.col('PAUV - T'),
        pl.col('PAUV').alias('PAUV - C'),
        pl.col('PAPB TEMP - T'),
        pl.col('PAPB TEMP').alias('PAPB TEMP - C'),
        pl.col('PAPB TIME - T'),
        pl.col('PAPB TIME').alias('PAPB TIME - C'),
        pl.col('BOOX TEMP'),
        pl.col('BOOX TIME'),
        # pl.col('ODFX Q2'),
        pl.col('SEUV') * 10, # change unit to mJ
        pl.col('SEPB TEMP'),
        pl.col('SEPB TIME'),
    )
    .write_excel(cwd / "eda_summary.xlsx")
)

In [8]:
qtime_raw = pl.read_excel(cwd / '6117 history-td.xlsx')

In [13]:
(
    qtime_raw
        .filter(
            pl.col("CF_GLASS_ID") != "NA",   
            pl.col("ARRAY_GLASS_ID") != "NA",   
        )
        .select("ARRAY_GLASS_ID", "CF_GLASS_ID")
        .unique()
        # .head()
        .write_excel(cwd / "tft_cf.xlsx")
)

In [60]:
(
    qtime_raw
        .filter(
            pl.col("COMPONENT_ID") == "611A2APB2HP212",
            pl.col('SUB_EQUIP_ID').str.contains(r"2CASM\d+"),
        )
        .head(20)
)

STEP_ID,STEP_SEQ,GLASS_ID,COMPONENT_ID,TRACK_IN_TIME,TRACK_OUT_TIME,COMPONENT_TYPE,PRODUCT_ID,BATCH_ID,LOT_ID,EQUIP_ID,ARRAY_LOT_ID,ARRAY_GLASS_ID,CF_GLASS_ID,RECIPE_ID,TA_ID,PLAN_ID,PLAN_ID_VERSION,SOURCE_CASSETTE_ID,SLOT_ID,FAB_AREA,SUB_EQUIP_ID,LOT_TYPE,BATCH_TYPE,CELL_PART_NO,OCF_VENDOR_LOT_ID,OCF_VENDOR_GLASS_ID,SAMPLING_FLAG,ABNORMAL_FLAG,PANEL_SIZE,MODEL_NAME,PROCESS_MODE,TA_SHIFT,Q_TAP,SUB_PRODUCT1,SUB_PRODUCT2,SUB_PRODUCT3,CHAMBER_PATH,CHAMBER_TIME_PATH,PI_REWORK_COUNT,REWORK_COUNT,RN
str,i64,str,str,datetime[ms],datetime[ms],i64,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64
"""6310_AS1_ARC""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 14:28:05,2026-01-22 14:38:07,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,null,"""T2""","""2CASM120""","""E""","""E""",""" """,""" """,""" """,""" """,""" ""","""6.1W""","""F061A17-6T1""","""IPS""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,982
"""6310_AS1_ARC_BF""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 14:54:36,2026-01-22 14:55:05,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,null,"""T2""","""2CASM140""","""E""","""E""",""" """,""" """,""" """,""" """,""" ""","""6.1W""","""F061A17-6T1""","""IPS""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,1008
"""6310_AS1_ARO""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 14:48:29,2026-01-22 14:54:36,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,null,"""T2""","""2CASM130""","""E""","""E""",""" """,""" """,""" """,""" """,""" ""","""6.1W""","""F061A17-6T1""","""IPS""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,1034
"""6310_AS1_LCD""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 15:11:49,2026-01-22 15:15:20,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,null,"""T2""","""2CASM1A0""","""E""","""E""",""" """,""" """,""" """,""" """,""" ""","""6.1W""","""F061A17-6T1""","""IPS""",""" """,""" """,""" """,""" """,""" ""","""2CASM1A3""","""20260122151439""",""" """,""" """,1060
"""6310_AS1_LCI""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 16:31:31,2026-01-22 17:24:50,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,null,"""T2""","""2CASM1E0""","""E""","""E""",""" """,""" """,""" """,""" """,""" ""","""6.1W""","""F061A17-6T1""","""IPS""",""" """,""" """,""" """,""" """,""" ""","""2CASM1E1""","""20260122172427""",""" """,""" """,1086
"""6310_AS1_LD""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 14:27:19,2026-01-22 14:28:05,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,50,"""T2""","""2CASM110""","""E""","""E""",""" """,""" """,""" """,""" """,""" ""","""6.1W""","""F061A17-6T1""","""IPS""",""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,""" """,1112
"""6310_AS1_ODF""",6310,"""611A2APB2HP212""","""611A2APB2HP212""",2026-01-22 15:15:20,2026-01-22 15:20:40,1,"""611A2C""","""EF261216""","""611A2YPB2HR2.03""","""2CASM100""","""611A2APB2HP2""","""611A2APB2HP212""","""611A2FPB2CDT47""","""611A_T01""","""EAP""","""IPSPHA1""",""" """,""" """,null,"""T2""","""2CASM1B0""","""E""","""E""",""

In [63]:
def qtime_get(
    qtime_raw: pl.DataFrame, 
    step: tuple[str, str], 
    filter: list[pl.Expr] | None = None,
) -> pl.DataFrame:
    result = (
        qtime_raw.filter(
            pl.col("STEP_ID").str.contains(step),
        )
        .select(
            pl.col("GLASS_ID").alias("ID"), 
            pl.col("TRACK_IN_TIME"),
            pl.col("TRACK_OUT_TIME"),
        )
    )
    
    if filter is not None:
        return result.filter(*filter)
    
    return result

In [65]:
t2_qtime_steps = {
    "PIPB": {"step": "6110_PI2_PIC", "filter": None},
    "PAUV": {"step": "6220_PH3_PA_UV", "filter": None},
    "PAPB": {"step": "6220_PH3_OVEN", "filter": None},
    "BOOX": {"step": "6310_AS1_ARO", "filter": None},
    "ODFX": {"step": "6310_AS1_LCD", "filter": None},
    "SEUV": {"step": "6310_AS1_UV_MA", "filter": None},
}
qtime = qtime_raw.select(pl.col("GLASS_ID").alias("ID")).unique()
for step, kwargs in t2_qtime_steps.items():
    qtime = qtime.join(
        qtime_get(qtime_raw, **kwargs)
        .select(
            "ID",
            pl.col("TRACK_IN_TIME").alias(step + "_IN"),
            pl.col("TRACK_OUT_TIME").alias(step + "_OUT"),
        ),
        on='ID',
        how='left',
    )
qtime

ID,PIPB_IN,PIPB_OUT,PAUV_IN,PAUV_OUT,PAPB_IN,PAPB_OUT,BOOX_IN,BOOX_OUT,ODFX_IN,ODFX_OUT,SEUV_IN,SEUV_OUT
str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms],datetime[ms]
"""611A2APB2HP218""",2026-01-21 19:41:43,2026-01-21 19:43:16,2026-01-22 00:45:01,2026-01-22 00:45:53,2026-01-22 00:47:43,2026-01-22 01:20:41,2026-01-22 15:14:53,2026-01-22 15:21:01,2026-01-22 15:58:30,2026-01-22 16:02:00,2026-01-22 16:07:07,2026-01-22 16:11:20
"""611A2APB2HP225""",2026-01-21 19:45:37,2026-01-21 19:47:11,2026-01-22 00:41:57,2026-01-22 00:42:42,2026-01-22 00:44:03,2026-01-22 01:17:00,2026-01-22 21:20:08,2026-01-22 21:26:16,2026-01-22 21:45:25,2026-01-22 21:49:52,2026-01-22 21:56:31,2026-01-22 22:00:40
"""611A2FPB2CBT25""",2026-01-21 14:38:29,2026-01-21 14:40:02,2026-01-22 00:56:45,2026-01-22 00:57:50,2026-01-22 00:59:57,2026-01-22 01:32:56,null,null,null,null,null,null
"""611A2APB2HP202""",2026-01-21 17:12:35,2026-01-21 17:14:08,2026-01-22 17:27:34,2026-01-22 17:28:43,2026-01-22 17:30:57,2026-01-22 18:03:54,2026-01-23 21:09:09,2026-01-23 21:15:17,2026-01-23 21:41:06,2026-01-23 21:45:03,2026-01-23 21:50:12,2026-01-23 21:54:37
"""611A2APB2HP213""",2026-01-21 19:07:33,2026-01-21 19:09:06,2026-01-22 00:47:00,2026-01-22 00:47:50,2026-01-22 00:49:32,2026-01-22 01:22:33,2026-01-22 11:26:52,2026-01-22 11:33:58,2026-01-22 11:53:51,2026-01-22 11:57:27,2026-01-22 12:02:35,2026-01-22 12:06:49
"""611A2FPB2C8T35""",2026-01-21 11:56:13,2026-01-21 11:57:36,2026-01-22 13:30:53,2026-01-22 13:31:47,2026-01-22 13:32:58,2026-01-22 14:06:01,null,null,null,null,null,null
"""611A2FPB2CET14""",2026-01-21 14:28:04,2026-01-21 14:29:38,2026-01-22 01:01:49,2026-01-22 01:02:41,2026-01-22 01:04:29,2026-01-22 01:37:29,null,null,null,null,null,null
"""611A2APB2HP219""",2026-01-21 19:42:22,2026-01-21 19:43:56,2026-01-22 00:44:04,2026-01-22 00:44:54,2026-01-22 00:46:44,2026-01-22 01:19:42,2026-01-22 15:08:17,2026-01-22 15:14:25,2026-01-22 15:46:14,2026-01-22 15:49:03,2026-01-22 15:54:08,2026-01-22 15:58:22
"""611A2APB2HP204""",2026-01-21 17:13:14,2026-01-21 17:14:52,2026-01-22 13:38:05,2026-01-22 13:45:25,2026-01-22 13:47:38,2026-01-22 14:20:46,2026-01-23 02:19:42,2026-01-23 02:25:50,2026-01-23 02:44:08,2026-01-23 02:47:27,2026-01-23 02:52:31,2026-01-23 02:56:43


In [73]:
(
    condition
        .join(
            qtime.select([col for col in qtime.columns if qtime[col].null_count() == 0]),
            left_on="CF",
            right_on='ID',
            suffix="_CF",
            maintain_order="left",
        )
        .join(
            qtime,
            left_on="TFT",
            right_on="ID",
            suffix="_TFT",
            maintain_order="left",
        )
        .select(
            "Condition", "TFT", "CF",
            (pl.col('PAPB_IN'    ) - pl.col('PIPB_OUT'    )).alias('PIPB to PAPB - C'),
            (pl.col('PAPB_IN_TFT') - pl.col('PIPB_OUT_TFT')).alias('PIPB to PAPB - T'),
            (pl.col('BOOX_IN'    ) - pl.col('PAPB_OUT'    )).alias('PAPB to BOOX - C'),
            (pl.col('BOOX_IN'    ) - pl.col('PAPB_OUT_TFT')).alias('PAPB to BOOX - T'),
            # (pl.col('ODFX_IN'    ) - pl.col('LCDP_OUT'    )).alias('LCDP to ODFX'    ),
            (pl.col('SEUV_IN'    ) - pl.col('ODFX_OUT'    )).alias('ODFX to SEUV'    ),
        )
        # .write_excel(cwd / "qtime_summary.xlsx")
)

Condition,TFT,CF,PIPB to PAPB - C,PIPB to PAPB - T,PAPB to BOOX - C,PAPB to BOOX - T,ODFX to SEUV
str,str,str,duration[ms],duration[ms],duration[ms],duration[ms],duration[ms]
"""LC-1098""","""611A2APB2HP221""","""611A2FPB2CBT15""",10h 25m 47s,5h 33s,15h 27m 48s,15h 44m 39s,6m 50s
"""LC-1098""","""611A2APB2HP208""","""611A2FPB2CBT23""",10h 32m 18s,4h 54m 42s,15h 21m 10s,15h 38m 1s,5m 10s
"""STD-首件""","""611A2APB2HP213""","""611A2FPB2C9T27""",10h 35m 21s,5h 40m 26s,9h 47m 35s,10h 4m 19s,5m 8s
"""STD""","""611A2APB2HP212""","""611A2FPB2CDT47""",10h 34m 3s,5h 41m 5s,13h 9m 12s,13h 25m 56s,5m 20s
"""STD""","""611A2APB2HP216""","""611A2FPB2CET58""",10h 29m 39s,5h 5m 6s,13h 17m 36s,13h 34m 24s,5m 6s
"""PAUV 260""","""611A2APB2HP204""","""611A2FPB2C8T35""",1d 1h 35m 22s,20h 32m 46s,12h 13m 41s,11h 58m 56s,5m 4s
"""PAUV 260""","""611A2APB2HP214""","""611A2FPB2CDT41""",1d 31m 42s,18h 31m 16s,12h 7m 5s,11h 59m 7s,5m 16s
"""PAUV 220""","""611A2APB2HP206""","""611A2FPB2CGT19""",20h 52m 2s,16h 8m 14s,12h 58m 22s,13h 4m,5m 23s
"""PAUV 220""","""611A2APB2HP211""","""611A2FPB2CBT32""",20h 49m 26s,16h 7m 35s,13h 5m 37s,13h 11m 15s,5m 7s


PI prebake 溫度, 時間  
PI postbake 溫度, 時間  
PAUV dosage  
PAPB 溫度, 時間  
ARO 溫度, 時間  
SEUV dosage  
SMO 溫度, 時間  